# 08 — Three methods: subgradient, prox-linear, proximal point

Implements the three stochastic algorithms from Section 5.1 of Davis & Drusvyatskiy (2019)
applied to the phase retrieval problem:

$$\min_{x \in \mathbb{R}^d} \frac{1}{m} \sum_{i=1}^m |\langle a_i, x \rangle^2 - b_i|$$

1. **Stochastic subgradient** — model (1.4): linear model from a subgradient
2. **Stochastic prox-linear** — model (1.5): linearize the inner map $c(x) = \langle a, x\rangle^2$
3. **Stochastic proximal point** — model (1.6): use the full function as the model

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.ModelBasedSolver import ModelBasedSolver
from src.problems.phase_retrieval import (
    SubgradientPhaseRetrieval,
    ProxLinearPhaseRetrieval,
    ProximalPointPhaseRetrieval,
)

## Phase retrieval setup (Section 5.1)

Generate Gaussian measurements, target on unit sphere.

In [ ]:
torch.manual_seed(42)
d, m = 10, 30  # smallest config from the paper

true_x = torch.randn(d)
true_x = true_x / torch.norm(true_x)

a_matrix = torch.randn(m, d)
b_vector = (a_matrix @ true_x) ** 2

population_data = torch.cat([a_matrix, b_vector.unsqueeze(1)], dim=1)
print(f"d={d}, m={m}, ||true_x||={torch.norm(true_x).item():.4f}")

## Run all three methods

In [ ]:
methods = {
    "Subgradient": SubgradientPhaseRetrieval(rho=2.0),
    "Prox-Linear": ProxLinearPhaseRetrieval(rho=2.0),
    "Proximal Point": ProximalPointPhaseRetrieval(rho=2.0),
}

T = 500
beta = 5.0
results = {}

for name, prob in methods.items():
    torch.manual_seed(42)
    x_init = torch.randn(d)
    x_init = x_init / torch.norm(x_init)

    solver = ModelBasedSolver(
        problem=prob,
        data=population_data,
        x_init=x_init,
        T=T,
        batch_size=1,
        beta=beta,
        log_every=50,
    )
    final_x = solver.run()
    results[name] = solver.history

    recon_err = min(
        torch.norm(final_x - true_x).item(),
        torch.norm(final_x + true_x).item(),  # sign ambiguity in phase retrieval
    )
    print(f"{name:16s} | final obj = {solver.history['obj_values'][-1]:.6f} | recon err = {recon_err:.4f}")
    print()

## Convergence comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, hist in results.items():
    ax.plot(hist["iterations"], hist["obj_values"], label=name)

ax.set_xlabel("Iteration")
ax.set_ylabel(r"$\varphi(x_t)$")
ax.set_title(f"Phase retrieval (d={d}, m={m}), β={beta}")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step-size sensitivity (paper Figure 3 style)

Run each method across multiple β values and compare final objective.

In [ ]:
# 20 step-size values (β⁻¹ equally spaced in [10⁻⁴, 1], i.e. β in [1, 10⁴])
inv_betas = torch.linspace(1e-4, 1.0, 20).tolist()
beta_values = [1.0 / ib for ib in inv_betas]

T_epochs = 100  # 100 passes through the data
T_iters = T_epochs * m
n_rounds = 5  # average over multiple random seeds

sensitivity = {name: [] for name in methods}

for beta_val in beta_values:
    for name, prob in methods.items():
        obj_sum = 0.0
        for seed in range(n_rounds):
            torch.manual_seed(seed)
            x_init = torch.randn(d)
            x_init = x_init / torch.norm(x_init)

            solver = ModelBasedSolver(
                problem=prob,
                data=population_data,
                x_init=x_init,
                T=T_iters,
                batch_size=1,
                beta=beta_val,
                log_every=T_iters + 1,  # only log at the end
            )
            final_x = solver.run()
            obj_sum += prob.population_objective(final_x, population_data)

        sensitivity[name].append(obj_sum / n_rounds)

print("Step-size sweep complete.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name in methods:
    ax.plot(inv_betas, sensitivity[name], marker="o", markersize=3, label=name)

# Initial functional error (dashed blue line as in paper)
init_obj = methods["Subgradient"].population_objective(
    torch.randn(d) / torch.norm(torch.randn(d)), population_data
)
ax.axhline(y=init_obj, color="blue", linestyle="--", alpha=0.5, label="Initial error")

ax.set_xlabel(r"$\beta^{-1}$ (step-size parameter)")
ax.set_ylabel("Function gap after 100 epochs")
ax.set_title(f"Step-size sensitivity (d={d}, m={m})")
ax.set_yscale("log")
ax.set_xscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()